[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/junlinghu/book-agents/blob/main/tutorial/14-planning.ipynb)

## Colab setup

On Google Colab, use the setup in `tutorial/common/colab.py`. A fresh Colab runtime does not have this repository, so clone the repo into the session first. Then, before the lesson cells, run `setup_colab` from that module (import it from `tutorial.common.colab` and call it, or open the file and run `setup_colab` from there). The helper installs the tutorial packages, checks the repo out under `/content/book-agents` when it is still missing, and copies a Colab secret named `OPENAI_API_KEY`. Local Jupyter and VS Code can skip it when you already have the repo.

An API key is required. Set `OPENAI_API_KEY` in Colab secrets (the key icon, secret name `OPENAI_API_KEY`) before you run `setup_colab`. Locally, put the same name in the repository-root `.env`. A missing key stops the notebook. The key is not printed.

Edits you make in Colab stay in that session. They do not push to GitHub.

# 14. Planning

A multi-step customer question gets an explicit plan before any tool runs.

`draft_plan` is one Chat Completions call with an empty tool list. It returns a short numbered list. `run_agent` prints that plan, attaches it to the system message, and only then starts perceive, reason, act, observe. The plan cannot add a tool. Charge, cancel, and email are not steps it is allowed to invent.

The customer is Maya. She wants a jar she can eat, and she wants to know if it ships to Ohio. Preference, the catalog, and shipping are separate steps. The plan names the order. The loop still executes only registered tools.

## Setup

Find the repo so `import tutorial` works, then load the key. `chat` calls the OpenAI Chat Completions API. The key is not printed.

In [ ]:
import sys
from pathlib import Path


def find_root():
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "tutorial" / "common" / "client.py").is_file():
            return folder
    raise RuntimeError("Open this notebook inside the book-agents repository.")


ROOT = find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tutorial.runtime import require_key

_, model = require_key()
print("model:", model)

## Plan, then act

Importing `tutorial.common.plan` turns the plan step on. The earlier tools are imported so the plan has something real to name.

In [ ]:
import tutorial.common.facts
import tutorial.common.catalog
import tutorial.common.memory
import tutorial.common.skills
import tutorial.common.plan  # drafts a plan before the tool loop

from tutorial.common.get_db import reset_db
from tutorial.common.harness import check, system_text
from tutorial.common.loop import run_agent
from tutorial.common.plan import plan_steps

reset_db()
question = "I'm Maya. Please recommend a jar I can eat, and tell me whether you can ship it to Ohio."
result = run_agent(question, system_text(), max_steps=8)
print("STEPS:")
for step in plan_steps(result["plan"]):
    print(step)
print("ANSWER:", result["text"])
steps = plan_steps(result["plan"])
log = "\n".join(result["tool_log"])
check(len(steps) >= 2, "the plan is an ordered list written before the tools")
check("charge_card" not in result["plan"], "the plan does not add a card charge")
check(bool(result["tool_log"]), "the loop ran tools after the plan")
check("get_preference" in log or "preference" in result["plan"].lower(), "Maya's preference is part of the work")
check("query_catalog" in log or "get_store_fact" in log, "store data was read while following the plan")